# Regional Inertia Estimation

This run's Inertia Estimator outputs (one mean and median inertia estimate per window), compared
with the ground-truth regional inertia: error reports and an hourly comparison plot.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.feather as pf

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / "notebook_outputs"
PLOT_OUTPUT_FILE = OUTPUT_DIR / "kf_hourly_plot.png"  # hourly comparison plot

# Payload times count from this instant (ParaEMT's default start; the Player data uses it too).
EPOCH = pd.Timestamp("2026-01-01 00:00:00")


print(f"Run directory: {RUN_DIR}")

## Load the recorded outputs

The run's `wiring.json` says which Recorder is attached to which estimator output, and which
Players feed the estimator, so component names don't matter. Times are converted to seconds.

In [ ]:
def find_run_parts(run_dir: Path):
    """Return the estimator component, {output port: feather path}, and the Players feeding it."""
    wiring = json.loads((run_dir / "wiring.json").read_text(encoding="utf-8"))
    components = {c["name"]: c for c in wiring["components"]}
    estimators = [
        c for c in wiring["components"]
        if {"region", "window_sec"} <= set(c.get("parameters", {}))
    ]
    if len(estimators) != 1:
        raise ValueError(f"Expected one Inertia Estimator in wiring.json; found {len(estimators)}.")
    estimator = estimators[0]

    recordings, players = {}, []
    for link in wiring["links"]:
        if link["source"] == estimator["name"] and components[link["target"]]["type"] == "Recorder":
            recorder = components[link["target"]]
            recordings[link["source_port"]] = (
                run_dir / "build" / recorder["name"] / recorder["parameters"]["feather_filename"]
            )
        elif link["target"] == estimator["name"] and "filename" in components[link["source"]]["parameters"]:
            players.append(components[link["source"]])
    return estimator, recordings, players


def load_output(recordings: dict[str, Path], port: str) -> pd.DataFrame:
    if port not in recordings:
        raise KeyError(f"No Recorder is attached to the estimator's '{port}' output. Add one and run again.")
    path = recordings[port]
    if not path.is_file():
        raise FileNotFoundError(f"{port}: {path} does not exist.")
    frame = pf.read_feather(path)
    frame["seconds"] = (pd.to_datetime(frame["time"]) - EPOCH).dt.total_seconds()
    return frame


estimator, recordings, players = find_run_parts(RUN_DIR)
REGION = estimator["parameters"]["region"]
WINDOW_SEC = float(estimator["parameters"]["window_sec"])

regional = load_output(recordings, "regional_signal")
inertia = load_output(recordings, "inertia").reset_index(drop=True)

if regional.empty or not regional["seconds"].is_monotonic_increasing:
    raise ValueError("regional_signal is empty or its time is not monotonic.")
# One regional sample per Player row; the last may be missing if the run stops one step early.
expected_rows = min(int(p["parameters"]["number_of_timesteps"]) for p in players) if players else None
if expected_rows is not None and len(regional) < expected_rows - 1:
    raise ValueError(f"regional_signal has {len(regional)} rows; the Players sent {expected_rows}.")
if inertia.empty:
    raise ValueError(
        f"No inertia estimate was published: no {WINDOW_SEC:g} s window closed "
        "(partial windows are discarded)."
    )

print(f"Region: {REGION}; window: {WINDOW_SEC:g} s")
print(f"Regional samples: {len(regional):,}; inertia estimates: {len(inertia)}")

## Ground truth

The ground-truth hourly regional inertia is kept beside the replayed data as `ground_truth.json`
and read here only; the estimator never sees it. Each estimate is matched to the snapshot whose start
precedes its window end.

In [ ]:
def load_ground_truth(players: list[dict], region: str) -> list[dict]:
    """Ground-truth snapshots as [{hour_num, start_sec, true_energy_mva_s}], or [] if none."""
    for player in players:
        path = Path(player["parameters"]["filename"]).expanduser().parent / "ground_truth.json"
        if not path.is_file():
            continue
        truth = json.loads(path.read_text(encoding="utf-8"))
        if truth.get("region") != region:
            continue
        if "hours" in truth:
            return truth["hours"]
        hour = int(truth.get("hour_case", "hour_1").rsplit("_", 1)[-1])
        return [{"hour_num": hour, "start_sec": 0.0, "true_energy_mva_s": truth["true_energy_mva_s"]}]
    return []


snapshots = sorted(load_ground_truth(players, REGION), key=lambda s: s["start_sec"])

# The error reports and plot below take an hourly table with these columns.
rows = []
for i, estimate in inertia.iterrows():
    match = [s for s in snapshots if s["start_sec"] - 1.0 <= estimate["seconds"]]
    snapshot = match[-1] if match else None
    rows.append({
        "hour_num": snapshot["hour_num"] if snapshot else i + 1,
        "mean_energy_mva_s": float(estimate["mean_energy_mva_s"]),
        "median_energy_mva_s": float(estimate["median_energy_mva_s"]),
        "true_energy_mva_s": snapshot["true_energy_mva_s"] if snapshot else np.nan,
    })
hourly_output = pd.DataFrame(rows)
HAS_TRUTH = hourly_output["true_energy_mva_s"].notna().all()
if not HAS_TRUTH:
    print(f"No ground_truth.json for region {REGION} next to the Player data.")

## Error analysis and hourly plot

Both error reports (estimate minus ground truth, for the mean and the median estimate) are
printed, then the hourly plot is shown. Here each "hour" is one estimator window.

In [ ]:
def _error_report(
    hourly_output: pd.DataFrame,
    estimate_column: str,
    comparison_name: str,
) -> dict:
    required_columns = {"hour_num", "true_energy_mva_s", estimate_column}
    missing_columns = required_columns - set(hourly_output.columns)
    if missing_columns:
        raise ValueError(f"Missing columns for error analysis: {sorted(missing_columns)}")

    comparison = hourly_output[
        ["hour_num", "true_energy_mva_s", estimate_column]
    ].dropna()
    if comparison.empty:
        raise ValueError(f"No valid rows for {comparison_name}.")

    errors = (
        comparison[estimate_column].to_numpy(dtype=float)
        - comparison["true_energy_mva_s"].to_numpy(dtype=float)
    )
    return {
        "comparison": comparison_name,
        "error_definition": "KF estimate - ground truth",
        "unit": "MVA*s",
        "sample_count": int(errors.size),
        "maximum_error": float(np.max(errors)),
        "minimum_error": float(np.min(errors)),
        "mae": float(np.mean(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(np.square(errors)))),
    }


def build_error_jsons(hourly_output: pd.DataFrame) -> tuple[str, str]:
    mean_report = _error_report(
        hourly_output,
        estimate_column="mean_energy_mva_s",
        comparison_name="ground_truth_vs_mean_kf_estimate",
    )
    median_report = _error_report(
        hourly_output,
        estimate_column="median_energy_mva_s",
        comparison_name="ground_truth_vs_median_kf_estimate",
    )
    return (
        json.dumps(mean_report, ensure_ascii=False, indent=2),
        json.dumps(median_report, ensure_ascii=False, indent=2),
    )

In [ ]:
def plot_hourly_results(
    hourly_output: pd.DataFrame,
    output_file: Path | None = PLOT_OUTPUT_FILE,
) -> tuple[plt.Figure, plt.Axes]:
    required_columns = {
        "hour_num",
        "true_energy_mva_s",
        "mean_energy_mva_s",
        "median_energy_mva_s",
    }
    missing_columns = required_columns - set(hourly_output.columns)
    if missing_columns:
        raise ValueError(f"Missing columns for plotting: {sorted(missing_columns)}")

    plot_data = hourly_output.dropna(subset=list(required_columns)).sort_values("hour_num")
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(
        plot_data["hour_num"],
        plot_data["mean_energy_mva_s"],
        "-o",
        linewidth=2.5,
        markersize=6,
        label="Mean KF Estimate",
    )
    ax.plot(
        plot_data["hour_num"],
        plot_data["median_energy_mva_s"],
        "-s",
        linewidth=2.5,
        markersize=6,
        label="Median KF Estimate",
    )
    ax.plot(
        plot_data["hour_num"],
        plot_data["true_energy_mva_s"],
        "-^",
        linewidth=2.5,
        markersize=6,
        label="Ground Truth",
    )
    ax.grid(True)
    ax.set_xticks(range(1, 25))
    ax.set_xlabel("Hour")
    ax.set_ylabel("Inertia Energy (MVA*s)")
    ax.set_title("Hourly Kalman-Filtered Inertia vs Ground Truth")
    ax.legend()
    fig.tight_layout()

    if output_file is not None:
        output_file.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_file, dpi=150, bbox_inches="tight")

    return fig, ax

In [ ]:
if HAS_TRUTH:
    mean_error_json, median_error_json = build_error_jsons(hourly_output)

    print("\nGround truth vs mean KF estimate:")
    print(mean_error_json)
    print("\nGround truth vs median KF estimate:")
    print(median_error_json)

    figure, _ = plot_hourly_results(hourly_output, output_file=PLOT_OUTPUT_FILE)
    display(figure)
    plt.close(figure)
else:
    print("No ground truth for this run, so no error reports or hourly plot.")